# 05 — ONNX Export and Parity Check

**Purpose:** Export trained sklearn models to ONNX and verify numerical parity.  
**Acceptance criterion:** max probability difference sklearn vs onnxruntime < 0.1% (0.001).  
Class prediction must be identical in 100% of val cases.

**This notebook runs BEFORE any Node.js integration (F5).**  
If parity fails here, Node.js metrics are invalid.

**Input:** `training/models/rf_final.pkl`, `training/models/if_final.pkl`, `training/splits/val.parquet`  
**Output:** `models/rf.onnx`, `models/if.onnx`, `training/results/parity_report.txt`

## 1. Load Trained Models

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

MODELS  = Path("../../training/models")
SPLITS  = Path("../../training/splits")
RESULTS = Path("../../training/results")
RESULTS.mkdir(parents=True, exist_ok=True)

rf  = joblib.load(MODELS / "rf_v1.pkl")
iso = joblib.load(MODELS / "if_v1.pkl")

import json
with open(MODELS / "if_v1_metadata.json") as f:
    if_meta = json.load(f)

THRESHOLD = if_meta["threshold"]
print(f"RF loaded  : {rf.n_estimators} estimators")
print(f"IF loaded  , threshold: {THRESHOLD:.4f}")

## 2. Export RF to ONNX

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

val  = pd.read_parquet(SPLITS / "val.parquet")
y_val = val.pop("label")
X_val = val

N_FEATURES   = X_val.shape[1]
TARGET_OPSET = 17  # compatible with onnxruntime-node

initial_type = [("float_input", FloatTensorType([None, N_FEATURES]))]

rf_onnx = convert_sklearn(
    rf,
    initial_types=initial_type,
    target_opset=TARGET_OPSET,
    options={id(rf): {"zipmap": False}},  # flat probability tensor
)

with open(MODELS / "rf.onnx", "wb") as f:
    f.write(rf_onnx.SerializeToString())

print(f"RF exported : {MODELS / 'rf.onnx'}")
print(f"N features  : {N_FEATURES}, opset: {TARGET_OPSET}")

## 3. Export IF to ONNX

In [ ]:
iso_onnx = convert_sklearn(
    iso,
    initial_types=initial_type,
    target_opset=TARGET_OPSET,
)

with open(MODELS / "if.onnx", "wb") as f:
    f.write(iso_onnx.SerializeToString())

print(f"IF exported : {MODELS / 'if.onnx'}")

## 4. Parity Check: Python sklearn vs onnxruntime

In [ ]:
import onnxruntime as rt

rf_sess = rt.InferenceSession(str(MODELS / "rf.onnx"))
if_sess = rt.InferenceSession(str(MODELS / "if.onnx"))

sample    = X_val.sample(1000, random_state=42).astype(np.float32)
sample_np = sample.values

# RF parity
sklearn_probs = rf.predict_proba(sample)
onnx_probs    = rf_sess.run(None, {"float_input": sample_np})[1]  # index 1 = probabilities

max_diff_rf = np.abs(sklearn_probs - onnx_probs).max()
print(f"RF max probability difference : {max_diff_rf:.6f}")
assert max_diff_rf < 0.001, (
    f"GATE FAILED: RF parity violation. Max diff={max_diff_rf:.6f} >= 0.001"
)

# IF parity
sklearn_scores = iso.decision_function(sample)
onnx_scores    = if_sess.run(None, {"float_input": sample_np})[0].flatten()

max_diff_if = np.abs(sklearn_scores - onnx_scores).max()
print(f"IF max score difference        : {max_diff_if:.6f}")
assert max_diff_if < 0.001, (
    f"GATE FAILED: IF parity violation. Max diff={max_diff_if:.6f} >= 0.001"
)

print("PARITY CHECK PASSED — both models cleared for Node.js integration.")

## 5. Parity Report

In [ ]:
import json

report = {
    "rf_onnx_path":      str(MODELS / "rf.onnx"),
    "if_onnx_path":      str(MODELS / "if.onnx"),
    "target_opset":      TARGET_OPSET,
    "n_features":        N_FEATURES,
    "parity_samples":    1000,
    "rf_max_prob_diff":  float(max_diff_rf),
    "if_max_score_diff": float(max_diff_if),
    "parity_passed":     bool(max_diff_rf < 0.001 and max_diff_if < 0.001),
    "threshold_if":      THRESHOLD,
}

with open(MODELS / "parity_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(f"Saved: {MODELS / 'parity_report.json'}")
print("This file must exist and parity_passed=true before")
print("Sebastián integrates the ONNX files into the npm library.")